In [20]:
# Environment
from dotenv import dotenv_values
import os
from dotenv import load_dotenv

# Data manipulation
import pandas as pd

# API
import requests
import json

# Date and time
from datetime import datetime
import time

# Google Cloud
from google.cloud import bigquery
from google.oauth2 import service_account


In [14]:

url = "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='01-01-2021'&@dataFinalCotacao='10-01-2024'&$top=2000&$format=json&$select=cotacaoCompra,cotacaoVenda,dataHoraCotacao"

response = requests.get(url)

data = response.json()

df = pd.DataFrame(data["value"])

In [15]:
df.head()

,cotacaoCompra,cotacaoVenda,dataHoraCotacao
0,5.1620,5.1626,2021-01-04 13:07:33.461
1,5.3263,5.3269,2021-01-05 13:11:14.045
2,5.3176,5.3182,2021-01-06 13:12:28.251
3,5.3427,5.3433,2021-01-07 13:11:33.564
4,5.3677,5.3683,2021-01-08 13:08:31.586


In [16]:
df.shape

(942, 3)

In [17]:
response = requests.get(url)
data = response.json()["value"]

print(len(data))

942


In [18]:
print(df["dataHoraCotacao"].min())
print(df["dataHoraCotacao"].max())

2021-01-04 13:07:33.461
2024-10-01 13:10:02.0


In [21]:
load_dotenv()

credentials = service_account.Credentials.from_service_account_file(
    os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
)

client = bigquery.Client(
    credentials=credentials,
    project=os.getenv("PROJECT_ID")
)

In [24]:
client = bigquery.Client(project="foreign-trade-brazil")

table_id = "foreign-trade-brazil.Fact.PTAX"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE" # Replace the existing table to avoid duplicate records.
)

job = client.load_table_from_dataframe(
    df,
    table_id,
    job_config=job_config
)

job.result()

print(f"{len(df)} records sent to {table_id}")

942 records sent to foreign-trade-brazil.Fact.PTAX
